In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="bTDt4MMeCrHvGr2pB2XQ")
project = rf.workspace("moon-tmhma").project("human-detection-yycva-h4aar")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Human-Detection-1 in yolo26:: 100%|██████████| 6563/6563 [00:00<00:00, 9284.46it/s] 


In [2]:
from ultralytics import YOLO
model = YOLO("yolo26s.pt")

In [4]:
model.train(
    data="Human-Detection-1/data.yaml",
    epochs=50,
    imgsz=512,
    batch=16,
    device=0 
)

Ultralytics 8.4.36 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3768MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Human-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ad1e427abd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [5]:
model = YOLO("runs/detect/train3/weights/best.pt")

metrics = model.val(
    data=r"Human-Detection-1/data.yaml",
    imgsz=512,
    device=0
)


Ultralytics 8.3.240  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 298.5169.0 MB/s, size: 69.4 KB)
val: Scanning C:\Users\raink\Desktop\Smart CCTV\Human-Detection-1\valid\labels.cache... 240 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 240/240 239.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.5it/s 6.0s0.3s
                   all        240        318      0.949       0.89      0.942      0.678
Speed: 3.0ms preprocess, 10.4ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to C:\Users\raink\Desktop\Smart CCTV\runs\detect\val


In [1]:
from ultralytics import YOLO
model = YOLO("runs/detect/train3/weights/best.pt")

In [7]:
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

mAP@0.5: 0.9417760784721945
mAP@0.5:0.95: 0.6777138500698893
Precision: 0.9489466501676591
Recall: 0.889937106918239


In [ ]:
import cv2
import cloudinary, cloudinary.uploader
import pymongo

cap = cv2.VideoCapture(0)

client = pymongo.MongoClient("")
db = client["Surveillance"]
collection = db["detections"]

cloudinary.config(
    cloud_name="",
    api_key="",
    api_secret=""
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        source=frame,
        imgsz=512,
        conf=0.25,
        device=0,
        verbose=False
    )

    result = results[0]

    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()

        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"Human {conf:.2f}",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )
        
        cv2.imwrite("temp.png",frame)
        result = cloudinary.uploader.upload("temp.png")
        doc = {"image_url": result["secure_url"]}
        collection.insert_one(doc)
    cv2.imshow("Human Detection", frame)
    

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()